# Notebook 4 — Visual Place Recognition & Scene Graph Retrieval

A robot navigating a building needs to know *where it is* given only its current
camera images — no GPS, no fiducials. This notebook covers **Visual Place Recognition (VPR)**:
matching a query image to a database of reference views, then retrieving the
corresponding portion of the semantic scene graph.

---

## Prerequisites

| Topic | Why it matters |
|-------|---------------|
| Embedding spaces | Features are points in a high-dimensional vector space |
| K-nearest neighbour search | Retrieval = find the closest database point |
| K-means clustering | VLAD uses cluster centres to encode patch features |
| Recall@k metric | Standard VPR evaluation protocol |

---

## Learning Objectives

1. Understand the VPR problem and why raw pixels fail.
2. Extract DINOv2 patch features from images.
3. Aggregate patch features with **VLAD** into a single compact descriptor.
4. Build a reference database and perform nearest-neighbour matching.
5. Retrieve the semantic scene subgraph for any query location.

## 1 · The Place Recognition Problem

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

np.random.seed(7)

# ── Left: robot trajectory ────────────────────────────────────────────────────
ax = axes[0]
t = np.linspace(0, 2*np.pi, 200)
traj_x = np.cos(t) + 0.05*np.random.randn(200)
traj_y = np.sin(t)*0.6 + 0.05*np.random.randn(200)
ax.plot(traj_x, traj_y, 'b-', lw=1.5, alpha=0.5, label='Trajectory')

db_idx = np.arange(0, 200, 10)
ax.scatter(traj_x[db_idx], traj_y[db_idx], c='blue', s=50, zorder=5, label='DB frames')

q_idx = 37
ax.scatter(traj_x[q_idx], traj_y[q_idx], c='red', s=120, marker='*', zorder=6, label='Query')

# GT match
gt_idx = db_idx[np.argmin(np.abs(db_idx - q_idx))]
ax.annotate('', xy=(traj_x[gt_idx], traj_y[gt_idx]),
            xytext=(traj_x[q_idx], traj_y[q_idx]),
            arrowprops=dict(arrowstyle='->', color='green', lw=2))
ax.text(traj_x[gt_idx]+0.05, traj_y[gt_idx]+0.05, 'GT match', fontsize=8, color='green')

ax.set_title('Task: find DB frame closest to query', fontsize=10, fontweight='bold')
ax.legend(fontsize=8); ax.set_aspect('equal'); ax.grid(True, alpha=0.2)

# ── Centre: naive pixel matching fails ───────────────────────────────────────
ax = axes[1]
ax.axis('off')
ax.set_title('Why pixel matching fails', fontsize=10, fontweight='bold')
lines = [
    'Same place, different:',
    '  • Lighting (day vs. night)',
    '  • Viewpoint (left/right)',
    '  • Season / appearance',
    '',
    'Pixel MSE is sensitive to all of these.',
    '',
    'Solution: learn an embedding φ(·)',
    'that is invariant to these factors.',
    '',
    'We use DINOv2 — a self-supervised',
    'vision transformer trained to be',
    'viewpoint and lighting robust.',
]
for i, l in enumerate(lines):
    ax.text(0.05, 0.95-i*0.075, l, transform=ax.transAxes, fontsize=9, va='top')

# ── Right: embedding space ────────────────────────────────────────────────────
ax = axes[2]
ax.set_title('Embedding space
(same place → close)', fontsize=10, fontweight='bold')
np.random.seed(3)
n_places = 6
for i in range(n_places):
    centre = np.random.randn(2) * 2
    pts = centre + np.random.randn(4, 2) * 0.15
    col = plt.cm.tab10(i/n_places)
    ax.scatter(pts[:,0], pts[:,1], c=[col]*4, s=60, alpha=0.8)
    ax.text(centre[0], centre[1], f'Place {i+1}', fontsize=7, ha='center',
            color=col, fontweight='bold')

ax.set_xlabel('Embedding dim 1'); ax.set_ylabel('Embedding dim 2')
ax.grid(True, alpha=0.2); ax.set_aspect('equal')

plt.suptitle('Visual Place Recognition (VPR) Problem', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2 · DINOv2 Patch Features

DINOv2 is a Vision Transformer (ViT) trained with self-distillation.
Instead of using the global `[CLS]` token, we use the **patch token grid** —
one feature vector per 14×14 pixel patch.

For a 224×224 input this gives a **16×16 grid** of 768-dimensional vectors,
capturing fine-grained spatial structure useful for matching.

```
Input image  (H, W, 3)
     ↓  patchify into 14×14 blocks
Patch grid   (H/14, W/14, 3, 14, 14)
     ↓  ViT transformer
Patch tokens (H/14 × W/14, 768)
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── Left: image grid ──────────────────────────────────────────────────────────
ax = axes[0]; ax.axis('off'); ax.set_title('① Image → 14×14 patch grid', fontsize=10, fontweight='bold')
H, W = 16, 16
grid = np.zeros((H, W, 3))
np.random.seed(0)
for i in range(H):
    for j in range(W):
        grid[i,j] = [0.2 + 0.5*np.sin(i/H*np.pi), 0.3+0.3*j/W, 0.4+0.3*np.cos(j/W*np.pi)]
ax.imshow(grid)
for i in range(H+1):
    ax.axhline(i-0.5, color='white', lw=0.5, alpha=0.5)
for j in range(W+1):
    ax.axvline(j-0.5, color='white', lw=0.5, alpha=0.5)
ax.text(0.5, -0.05, f'{H}×{W} patches
each 14×14 pixels', ha='center',
        transform=ax.transAxes, fontsize=8)

# ── Centre: ViT attention ─────────────────────────────────────────────────────
ax = axes[1]; ax.axis('off'); ax.set_title('② ViT — self-attention across patches', fontsize=10, fontweight='bold')
# Show simplified attention pattern
attn = np.zeros((8,8))
for i in range(8):
    for j in range(8):
        attn[i,j] = np.exp(-((i-3)**2+(j-4)**2)/4)
im = ax.imshow(attn, cmap='hot', aspect='auto', extent=[-0.5,7.5,-0.5,7.5])
ax.set_title('② ViT attention heatmap
(one head, one query)', fontsize=10, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.7, label='attention weight')

# ── Right: patch feature vectors ─────────────────────────────────────────────
ax = axes[2]
np.random.seed(1)
D = 768
n_show = 6
feat_mat = np.random.randn(n_show, D)
feat_mat /= np.linalg.norm(feat_mat, axis=1, keepdims=True)  # L2 norm
im = ax.imshow(feat_mat, cmap='RdBu', aspect='auto', vmin=-0.1, vmax=0.1)
ax.set_yticks(range(n_show)); ax.set_yticklabels([f'patch {i}' for i in range(n_show)])
ax.set_xlabel('Feature dimension (768)')
ax.set_title('③ Patch feature matrix
(6 of H/14 × W/14 patches shown)', fontsize=10, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.7)

plt.suptitle('DINOv2 Patch Feature Extraction', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3 · VLAD Aggregation

16×16 = 256 patch vectors per image is too many for efficient retrieval.
**VLAD** (Vector of Locally Aggregated Descriptors) compresses them into a
single fixed-size descriptor.

### Steps

1. **K-means** on patch features from all DB images → $K$ cluster centres $\{\mathbf{c}_k\}$
2. For each image, assign each patch $\mathbf{x}_i$ to its nearest centre $k(i)$
3. Accumulate **residuals** within each cluster:
   $\mathbf{v}_k = \sum_{i:\,k(i)=k} (\mathbf{x}_i - \mathbf{c}_k)$
4. Concatenate all cluster residuals: $\mathbf{v} = [\mathbf{v}_1 \| \cdots \| \mathbf{v}_K]$
5. L2-normalise $\mathbf{v}$ → final descriptor of size $K \times D$

> With $K=32$ clusters and $D=768$, each image → a $24{,}576$-dimensional vector.
> This captures *where* in feature space the image's patches deviate from the average.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

np.random.seed(42)
D = 2   # 2-D for visualisation (real system uses 768-D)
K = 5

# Simulate patch features from two images
patches_img1 = np.vstack([
    np.random.randn(20, D) * 0.25 + [0.8,  0.5],
    np.random.randn(15, D) * 0.25 + [-0.5, 0.9],
    np.random.randn(10, D) * 0.25 + [0.0, -0.8],
])
patches_img2 = np.vstack([
    np.random.randn(20, D) * 0.25 + [0.7,  0.6],
    np.random.randn(15, D) * 0.25 + [-0.4, 0.8],
    np.random.randn(10, D) * 0.25 + [0.1, -0.7],
])

# K-means centres (pre-computed from all DB patches)
centers = np.array([
    [0.8,  0.5], [-0.5, 0.9], [0.0, -0.8], [-0.8, -0.3], [0.5, -0.5]
])

# ── Step 1: K-means centres ────────────────────────────────────────────────────
ax = axes[0]
palette = plt.cm.Set1(np.linspace(0,1,K))
all_patches = np.vstack([patches_img1, patches_img2])
# Assign to nearest centre
dists = np.linalg.norm(all_patches[:,None] - centers[None], axis=-1)
assigns = dists.argmin(axis=1)
for k in range(K):
    mask = assigns == k
    ax.scatter(all_patches[mask, 0], all_patches[mask, 1],
               c=[palette[k]], alpha=0.4, s=20)
    ax.scatter(*centers[k], c=[palette[k]], s=150, marker='*', edgecolors='black', lw=0.5, zorder=5)
ax.set_title('① K-means cluster assignment
(* = cluster centres)', fontsize=10, fontweight='bold')
ax.set_xlabel('dim 1'); ax.set_ylabel('dim 2'); ax.grid(True, alpha=0.3)

# ── Step 2: residuals ─────────────────────────────────────────────────────────
ax = axes[1]
dists1 = np.linalg.norm(patches_img1[:,None] - centers[None], axis=-1)
assigns1 = dists1.argmin(axis=1)
for k in range(K):
    mask = assigns1 == k
    pts = patches_img1[mask]
    ax.scatter(pts[:,0], pts[:,1], c=[palette[k]], alpha=0.6, s=20)
    ax.scatter(*centers[k], c=[palette[k]], s=150, marker='*', edgecolors='black', lw=0.5, zorder=5)
    for p in pts[:5]:
        ax.annotate('', xy=p, xytext=centers[k],
                    arrowprops=dict(arrowstyle='->', color=palette[k][:3], lw=0.8, alpha=0.6))
ax.set_title('② Residuals  $(\mathbf{x}_i - \mathbf{c}_k)$
per cluster', fontsize=10, fontweight='bold')
ax.set_xlabel('dim 1'); ax.set_ylabel('dim 2'); ax.grid(True, alpha=0.3)

# ── Step 3: aggregated VLAD vectors ──────────────────────────────────────────
ax = axes[2]
vlads = []
for img_patches in [patches_img1, patches_img2]:
    d = np.linalg.norm(img_patches[:,None] - centers[None], axis=-1)
    a = d.argmin(axis=1)
    v = np.zeros((K, D))
    for k in range(K):
        mask = a == k
        if mask.any():
            v[k] = (img_patches[mask] - centers[k]).sum(axis=0)
    v /= max(np.linalg.norm(v), 1e-12)
    vlads.append(v.flatten())

vlads = np.array(vlads)
im = ax.imshow(vlads, cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
ax.set_yticks([0,1]); ax.set_yticklabels(['Image 1', 'Image 2'])
ax.set_xlabel(f'VLAD dimension  (K×D = {K}×{D} = {K*D})')
ax.set_title('③ VLAD descriptors
(concatenated + L2-normalised)', fontsize=10, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)

# Similarity
sim = vlads[0] @ vlads[1]
ax.text(0.5, -0.18, f'Cosine similarity(img1, img2) = {sim:.3f}',
        transform=ax.transAxes, ha='center', fontsize=10, color='darkblue')

plt.suptitle('VLAD Aggregation Pipeline', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4 · Building the Reference Database

In [ ]:
import os
import pickle
import numpy as np
from spot_semantic_mapping.configs.loader import cfg
from spot_semantic_mapping.models.embedding import DinoModel
from spot_semantic_mapping.localization.encoder import ImageEncoder

IPHONE_PATH = '../data/iphone/3578aa5730'
DB_PICKLE   = '../outputs/vpr_db.pkl'

dino    = DinoModel(cfg)
encoder = ImageEncoder(dino)

poses = load_poses(os.path.join(IPHONE_PATH, 'odometry.csv'))
import skvideo.io
all_frames = list(skvideo.io.vreader(os.path.join(IPHONE_PATH, 'rgb.mp4')))

# Sample every 10th frame as database
DB_STEP = 10
db_frames  = all_frames[::DB_STEP]
db_poses   = poses[::DB_STEP]
db_images  = [Image.fromarray(f) for f in db_frames]

print(f'Database size: {len(db_images)} frames (every {DB_STEP}th of {len(all_frames)})')

# Embed with VLAD
db_emb = encoder.embed(
    db_images,
    patches     = True,
    agg_method  = cfg.vpr['agg_method'],
    num_clusters= cfg.vpr['num_clusters'],
    save_path   = DB_PICKLE,
    save        = True,
    grayscale   = False,
)
print(f'DB embeddings shape: {db_emb.shape}   ({db_emb.shape[1]:,} dims)')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualise the embedding space using PCA
from sklearn.decomposition import PCA

pca = PCA(n_components=2).fit(db_emb)
coords = pca.transform(db_emb)
db_positions = np.array([T[:3, 3] for T in db_poses])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Colour by progression through sequence
t = np.arange(len(db_emb))
sc = axes[0].scatter(coords[:,0], coords[:,1], c=t, cmap='viridis', s=40, alpha=0.8)
plt.colorbar(sc, ax=axes[0], label='Frame index')
axes[0].set_title('DB embeddings — PCA projection\n(colour = frame order)', fontsize=10, fontweight='bold')
axes[0].set_xlabel('PC 1'); axes[0].set_ylabel('PC 2'); axes[0].grid(True, alpha=0.3)

# Trajectory coloured the same way
sc2 = axes[1].scatter(db_positions[:,0], db_positions[:,2], c=t, cmap='viridis', s=40)
plt.colorbar(sc2, ax=axes[1], label='Frame index')
axes[1].set_title('Physical trajectory (XZ top-down)\n(same colour scheme)', fontsize=10, fontweight='bold')
axes[1].set_xlabel('X (m)'); axes[1].set_ylabel('Z (m)')
axes[1].set_aspect('equal'); axes[1].grid(True, alpha=0.3)

plt.suptitle('Reference Database: Embeddings vs. Physical Positions', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5 · Query Localisation

In [ ]:
from PIL import Image
from spot_semantic_mapping.localization.localizer import localize

# Use a held-out query frame (not in DB)
QUERY_IDX = 73
query_rgb   = all_frames[QUERY_IDX]
query_pose  = poses[QUERY_IDX]
query_image = Image.fromarray(query_rgb)

# Embed the query
with open(DB_PICKLE, 'rb') as f:
    db_data = pickle.load(f)
db_emb_loaded = db_data['embeddings']

query_emb = encoder.embed(
    [query_image],
    patches      = True,
    agg_method   = cfg.vpr['agg_method'],
    num_clusters = cfg.vpr['num_clusters'],
    save         = False,
    save_path    = DB_PICKLE,   # reuse cluster centres from DB
)

# Retrieve top-k matches
scores, sorted_idx = localize(query_emb, db_emb_loaded)
top_k = 5
print(f'Query frame {QUERY_IDX}')
print(f'Top-{top_k} retrievals:')
for rank, idx in enumerate(sorted_idx[0, :top_k]):
    db_frame_idx = idx * DB_STEP
    dist = np.linalg.norm(
        query_pose[:3,3] - db_poses[idx][:3,3]
    )
    print(f'  Rank {rank+1}: DB[{idx}] (frame {db_frame_idx})  '
          f'score={scores[0,idx]:.4f}  physical dist={dist:.2f} m')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

# Row 0: query + top-3 matches
axes[0,0].imshow(query_rgb)
axes[0,0].set_title(f'Query frame {QUERY_IDX}', fontsize=10, fontweight='bold',
                    color='red')
axes[0,0].axis('off')

for col, rank in enumerate([0, 1, 2]):
    idx = sorted_idx[0, rank]
    db_frame_idx = idx * DB_STEP
    match_rgb = all_frames[db_frame_idx]
    axes[0, col+1 if col < 2 else 2].imshow(match_rgb) if col < 2 else None
    if col < 2:
        dist = np.linalg.norm(query_pose[:3,3] - db_poses[idx][:3,3])
        axes[0, col+1].set_title(f'Rank {rank+1}  (DB frame {db_frame_idx})
'
                                  f'score={scores[0,idx]:.3f}  dist={dist:.2f} m',
                                  fontsize=9)
        axes[0, col+1].axis('off')

axes[0, 3 if False else 0]  # unused, already filled

# Row 1: trajectory + score bar
ax_traj = axes[1, 0]
db_pos = np.array([T[:3,3] for T in db_poses])
q_pos  = query_pose[:3,3]
ax_traj.plot(db_pos[:,0], db_pos[:,2], 'b.', ms=6, alpha=0.5, label='DB frames')
top_pos = db_poses[sorted_idx[0,0]][:3,3]
ax_traj.scatter(*q_pos[[0,2]],    color='red',   s=150, marker='*', zorder=5, label='Query')
ax_traj.scatter(*top_pos[[0,2]],  color='green', s=100, marker='s', zorder=5, label='Top-1')
ax_traj.plot([q_pos[0], top_pos[0]], [q_pos[2], top_pos[2]], 'g--', lw=1.5)
ax_traj.set_title('Localisation on Trajectory', fontsize=10, fontweight='bold')
ax_traj.set_xlabel('X (m)'); ax_traj.set_ylabel('Z (m)')
ax_traj.legend(fontsize=8); ax_traj.grid(True, alpha=0.3); ax_traj.set_aspect('equal')

# Score distribution
ax_score = axes[1, 1]
score_vals = scores[0, sorted_idx[0]]
ax_score.bar(range(len(score_vals)), score_vals, color=['green']+['steelblue']*(len(score_vals)-1), alpha=0.8)
ax_score.axhline(score_vals[0]*0.9, color='orange', ls='--', lw=1.5, label='90% of top-1')
ax_score.set_xlabel('Retrieval rank'); ax_score.set_ylabel('Cosine similarity')
ax_score.set_title('Similarity Score Distribution
(descending)', fontsize=10, fontweight='bold')
ax_score.legend(fontsize=8); ax_score.grid(True, alpha=0.3)

axes[1, 2].axis('off')

plt.suptitle(f'VPR Retrieval — Query frame {QUERY_IDX}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6 · Scene Graph Retrieval

In [ ]:
from spot_semantic_mapping.localization.localizer import retrieve_subgraphs, print_subgraph
import json

GRAPH_PATH = '../outputs/scene_graph_nb03/scene_graph.json'
with open(GRAPH_PATH) as f:
    g = json.load(f)

# Retrieve the subgraph visible from the top-k matched positions
subgraph = retrieve_subgraphs(
    dataset       = {'db_poses': db_poses},
    ordered_indices = sorted_idx[0, :top_k],
    g             = g,
    top_k         = top_k,
    window        = cfg.vpr.get('window_m', 3.0),
)

print('Subgraph at query location:')
print(print_subgraph(subgraph))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import numpy as np

if subgraph['nodes'] is not None and len(subgraph['nodes']) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # ── Left: full graph, subgraph highlighted ────────────────────────────────
    ax = axes[0]
    all_nodes = g['nodes']
    sub_ids = set(subgraph['nodes']['oid'].tolist())

    xs = [n['centroid'][0] for n in all_nodes]
    zs = [n['centroid'][2] for n in all_nodes]
    cols = ['#E74C3C' if n['oid'] in sub_ids else '#AED6F1' for n in all_nodes]
    ax.scatter(xs, zs, c=cols, s=80, zorder=5)
    for n, c in zip(all_nodes, cols):
        ax.text(n['centroid'][0]+0.02, n['centroid'][2]+0.02,
                f"[{n['oid']}]{n['label'][:8]}", fontsize=6,
                color='darkred' if n['oid'] in sub_ids else 'grey')

    q_pos = query_pose[:3,3]
    ax.scatter(q_pos[0], q_pos[2], c='gold', s=200, marker='*', zorder=6, label='Query pos')
    circle = plt.Circle((q_pos[0], q_pos[2]), cfg.vpr.get('window_m',3.0),
                         fill=False, color='red', ls='--', lw=1.5, label=f'Retrieval window')
    ax.add_patch(circle)

    legend_handles = [
        mpatches.Patch(color='#E74C3C', label=f'Subgraph nodes ({len(sub_ids)})'),
        mpatches.Patch(color='#AED6F1', label='Other nodes'),
        plt.scatter([], [], c='gold', marker='*', s=100, label='Query position'),
    ]
    ax.legend(handles=legend_handles[:2] + [plt.Line2D([],[], marker='*',
              color='w', markerfacecolor='gold', markersize=12, label='Query')],
              fontsize=8)
    ax.set_title('Full Scene Graph — Subgraph Highlighted', fontsize=10, fontweight='bold')
    ax.set_xlabel('X (m)'); ax.set_ylabel('Z (m)')
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

    # ── Right: retrieved subgraph only ────────────────────────────────────────
    ax = axes[1]
    sub_nodes_df = subgraph['nodes']
    sub_edges_df = subgraph.get('edges')

    G2 = nx.DiGraph()
    for _, row in sub_nodes_df.iterrows():
        G2.add_node(row['oid'], label=str(row.get('label',''))[:10])
    if sub_edges_df is not None:
        for _, row in sub_edges_df.iterrows():
            if row['src_id'] in sub_ids and row['dst_id'] in sub_ids:
                G2.add_edge(row['src_id'], row['dst_id'], rel=str(row.get('relation','')))

    pos2 = nx.spring_layout(G2, seed=0, k=2.5)
    nx.draw_networkx_nodes(G2, pos2, ax=ax, node_color='#E74C3C', node_size=500, alpha=0.9)
    nx.draw_networkx_labels(G2, pos2, ax=ax,
        labels={n: f"{n}:{G2.nodes[n].get('label','')}" for n in G2.nodes()}, font_size=7)
    nx.draw_networkx_edges(G2, pos2, ax=ax, arrows=True, edge_color='grey', alpha=0.7,
                           connectionstyle='arc3,rad=0.15')
    if sub_edges_df is not None:
        edge_lbl = {(r['src_id'],r['dst_id']): r.get('relation','') 
                    for _,r in sub_edges_df.iterrows() 
                    if r['src_id'] in sub_ids and r['dst_id'] in sub_ids}
        nx.draw_networkx_edge_labels(G2, pos2, edge_lbl, ax=ax, font_size=6.5)
    ax.set_title(f'Retrieved Subgraph ({len(G2.nodes())} nodes)', fontsize=10, fontweight='bold')
    ax.axis('off')

    plt.suptitle('Scene Graph Retrieval from Query Location', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No subgraph nodes found — try increasing the window_m parameter.')

## Summary

| Stage | What happens | Key parameter |
|-------|-------------|---------------|
| DINOv2 patch features | ViT produces H/14 × W/14 × 768 tokens | — |
| VLAD aggregation | K-means on patches → residual sum → L2 norm | `num_clusters` (32) |
| DB construction | Every $k$-th frame embedded + stored | `db_step` |
| Retrieval | Query VLAD → cosine similarity → top-k | `top_k` |
| Subgraph retrieval | Top-k positions → objects within `window_m` metres | `window_m` |

### What you have learned across all 4 notebooks

```
Notebook 1 → Config, camera model, SE(3) poses, RGB-D loading
Notebook 2 → YOLO detection, SAM2 masks, SigLIP embeddings
Notebook 3 → 3-D back-projection, ObjectTracker3D, scene graph export
Notebook 4 → DINOv2 patches, VLAD, VPR retrieval, subgraph query
```

The full system forms a closed loop:
**sense → detect → lift to 3-D → track → enrich → localise → retrieve**.